## Predictive Maintenance: Remaining Useful Life (RUL) Prediction

The aim of this notebook is to predict **Remaining Useful Life (RUL) in hours** using machine sensor, environmental, and maintenance-related features.  
This is a **regression problem**, where the target variable is `rul_hours`.

In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

## 1. Load the Dataset

In [ ]:
# Load the dataset
data = pd.read_csv("D:\ML Assignment\data\predictive_maintenance_v3.csv")

# Preview the first few rows
data.head()

## 2. Initial Data Understanding

In [ ]:
# Shape of the dataset
data.shape

In [ ]:
# Column names
data.columns.tolist()

In [ ]:
# Basic dataset information
data.info()

In [ ]:
# Summary statistics for numerical columns
data.describe()

In [ ]:
# Percentage of missing values in each column
missing_pct = (data.isnull().sum() / len(data) * 100).sort_values(ascending=False)
missing_pct[missing_pct > 0]

In [ ]:
# Duplicate row count
data.duplicated().sum()

### Observation
The dataset contains both numerical and categorical variables, along with timestamp and machine identifier columns.  
A small proportion of missing values is present in some sensor-related numerical features, which should be handled before modelling.

## 3. Data Type Conversion

In [ ]:
# Convert timestamp to datetime format
data["timestamp"] = pd.to_datetime(data["timestamp"], errors="coerce")

# Confirm updated data types
data.dtypes

## 4. Exploratory Data Analysis (EDA)

### 4.1 Distribution of the Target Variable

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data["rul_hours"], bins=40)
plt.title("Distribution of RUL Hours")
plt.xlabel("RUL Hours")
plt.ylabel("Frequency")
plt.savefig("Distribution of rul hours")
plt.show()

### Observation
The target variable is right-skewed, with many machines having lower RUL values and fewer observations at the higher end.  
This suggests the prediction task is challenging, especially for machines with large remaining life.

### 4.2 Machine Type vs RUL

In [ ]:
data.groupby("machine_type")["rul_hours"].describe()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x="machine_type", y="rul_hours", data=data, palette="Set2")
plt.title("Machine Type vs Remaining Useful Life (RUL)")
plt.xlabel("Machine Type")
plt.ylabel("RUL (hours)")
plt.savefig("machine type vs rul")
plt.show()

### Observation
RUL varies across machine types. Compressors generally show higher median RUL, while robotic arms tend to have lower values.  
However, the distributions overlap considerably, which suggests that machine type alone is not enough to explain remaining useful life.

### 4.3 Vibration RMS vs RUL

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x="vibration_rms", y="rul_hours", data=data, alpha=0.4)
plt.title("Vibration RMS vs Remaining Useful Life (RUL)")
plt.xlabel("Vibration RMS")
plt.ylabel("RUL (hours)")
plt.savefig("vibration rms vs rul")
plt.show()

### Observation
Higher vibration levels are generally associated with lower RUL.  
At the same time, low vibration does not always guarantee high RUL, which means vibration is useful but not sufficient on its own.

### 4.4 Hours Since Maintenance vs RUL

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x="hours_since_maintenance", y="rul_hours", data=data, alpha=0.4)
plt.title("Hours Since Maintenance vs Remaining Useful Life (RUL)")
plt.xlabel("Hours Since Maintenance")
plt.ylabel("RUL (hours)")
plt.savefig("hours since maintenance vs rms")
plt.show()

### Observation
As the hours since maintenance increase, the RUL generally decreases, reflecting the expected wear of machines over time.  
The pattern is not perfectly linear, which indicates that maintenance history should be combined with other sensor features for prediction.

### 4.5 Motor Temperature vs RUL

In [ ]:
plt.figure(figsize=(8, 5))
sns.regplot(
    x="temperature_motor",
    y="rul_hours",
    data=data,
    scatter_kws={"alpha": 0.25},
    line_kws={"color": "red"}
)
plt.title("Motor Temperature vs RUL (Trend)")
plt.xlabel("Motor Temperature")
plt.ylabel("RUL (hours)")
plt.show()

### Observation
Motor temperature shows a negative relationship with RUL, meaning higher temperatures are generally linked to faster degradation.  
Still, the spread in the plot suggests temperature should be interpreted together with other machine health indicators.

## 5. Data Cleaning and Preprocessing

### 5.1 Handle Missing Values

In [ ]:
# Numerical sensor columns with missing values
sensor_cols = [
    "vibration_rms",
    "temperature_motor",
    "current_phase_avg",
    "pressure_level",
    "rpm"
]

# Group-wise imputation based on machine type
for col in sensor_cols:
    data[col] = data.groupby("machine_type")[col].transform(lambda x: x.fillna(x.median()))
    data[col] = data[col].fillna(data[col].median())  # fallback if any NaNs remain

# Check remaining missing values
data[sensor_cols].isnull().sum()

### Observation
Missing values were filled using **group-wise median imputation by machine type**.  
This approach keeps machine-specific patterns more effectively than using a single global value for all machines.

### 5.2 Remove Leakage and Identifier Columns

In [ ]:
# Drop columns that should not be used for RUL prediction
data = data.drop(columns=[
    "failure_within_24h",
    "failure_type",
    "estimated_repair_cost"
])

# Drop identifier columns after EDA
data = data.drop(columns=["timestamp", "machine_id"])

data.head()

### Observation
Failure-related fields and repair cost were removed because they contain future information and would cause data leakage.  
Timestamp and machine ID were also excluded from modelling because they are identifiers rather than predictive features in this tabular setup.

### 5.3 Outlier Handling

In [ ]:
# Numerical columns for outlier handling
num_cols = [
    "vibration_rms",
    "temperature_motor",
    "current_phase_avg",
    "pressure_level",
    "rpm",
    "hours_since_maintenance",
    "ambient_temp"
]

def cap_outliers_iqr(df, cols):
    df = df.copy()
    for col in cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        df[col] = df[col].clip(lower, upper)
    return df

data = cap_outliers_iqr(data, num_cols)
data.head()

### Observation
Outliers were capped using the IQR method instead of removing rows.  
This reduces the influence of extreme values while preserving the overall size of the dataset.

### 5.4 Skewness Check

In [ ]:
data[num_cols].skew().sort_values(ascending=False)

### Observation
Most numerical features show only mild to moderate skewness.  
Since tree-based models are robust to such distributions, no additional transformations were applied.

## 6. Feature and Target Split

In [ ]:
X = data.drop("rul_hours", axis=1)
y = data["rul_hours"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

## 7. Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

### Observation
The data was split into training and testing sets using an 80:20 ratio.  
This allows the models to be trained on one portion of the dataset and evaluated on unseen data.

## 8. Encoding and Scaling

In [ ]:
# Separate categorical and numerical columns
categorical_cols = ["machine_type", "operating_mode"]
numerical_cols = [col for col in X_train.columns if col not in categorical_cols]

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

In [ ]:
# One-hot encode categorical variables
X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# Align train and test columns
X_train_encoded, X_test_encoded = X_train_encoded.align(
    X_test_encoded, join="left", axis=1, fill_value=0
)

print("Encoded train shape:", X_train_encoded.shape)
print("Encoded test shape :", X_test_encoded.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Scale only the numerical columns for models that need scaling
scaler = StandardScaler()

X_train_scaled = X_train_encoded.copy()
X_test_scaled = X_test_encoded.copy()

X_train_scaled[numerical_cols] = scaler.fit_transform(X_train_encoded[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test_encoded[numerical_cols])

X_train_scaled.head()

### Observation
Categorical variables were converted using one-hot encoding, while numerical features were scaled only for models sensitive to feature magnitude.  
Tree-based models were trained on the encoded but unscaled data.

## 9. Model Training and Evaluation

In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

In [ ]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2 = r2_score(true, predicted)
    return mae, rmse, r2

In [ ]:
def train_scaled_models(X_train, X_test, y_train, y_test):
    models = {
        "Linear Regression": LinearRegression(),
        "Lasso": Lasso(),
        "Ridge": Ridge(),
        "KNN": KNeighborsRegressor()
    }

    results = []

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mae, rmse, r2 = evaluate_model(y_test, y_pred)
        cv_r2 = cross_val_score(model, X_train, y_train, cv=5, scoring="r2").mean()

        results.append({
            "Model": name,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "CV_R2": cv_r2
        })

    return pd.DataFrame(results)

In [ ]:
def train_tree_models(X_train, X_test, y_train, y_test):
    models = {
        "Decision Tree": DecisionTreeRegressor(random_state=42),
        "Random Forest": RandomForestRegressor(random_state=42),
        "XGBoost": XGBRegressor(random_state=42),
        "CatBoost": CatBoostRegressor(verbose=False, random_state=42),
        "AdaBoost": AdaBoostRegressor(random_state=42),
        "LGBM Regressor": LGBMRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=-1,
            num_leaves=31,
            random_state=42
        )
    }

    results = []

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mae, rmse, r2 = evaluate_model(y_test, y_pred)
        cv_r2 = cross_val_score(model, X_train, y_train, cv=5, scoring="r2").mean()

        results.append({
            "Model": name,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "CV_R2": cv_r2
        })

    return pd.DataFrame(results)

In [ ]:
# Train models
scaled_results = train_scaled_models(X_train_scaled, X_test_scaled, y_train, y_test)
tree_results = train_tree_models(X_train_encoded, X_test_encoded, y_train, y_test)

# Combine and sort results
final_results = pd.concat([scaled_results, tree_results], ignore_index=True)
final_results = final_results.sort_values(by="R2", ascending=False)

final_results

### Observation
Tree-based ensemble models perform much better than linear and distance-based models.  
Random Forest achieved the best overall performance, which indicates that the relationship between the features and RUL is largely non-linear.

## 10. Feature Importance of the Best Model

In [ ]:
# Train the best baseline model again for feature importance
best_rf = RandomForestRegressor(random_state=42)
best_rf.fit(X_train_encoded, y_train)

feature_importance = pd.DataFrame({
    "Feature": X_train_encoded.columns,
    "Importance": best_rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance.head(10)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=feature_importance.head(10),
    x="Importance",
    y="Feature",
    palette="viridis"
)
plt.title("Top 10 Important Features - Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

### Observation
The feature importance plot highlights which sensor and operational variables contribute most to RUL prediction.  
This helps explain the model behaviour and provides useful insight into machine degradation patterns.

## 11. Advanced Experiment: Optuna Hyperparameter Tuning for Random Forest

In [ ]:
import optuna
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Cross-validation strategy
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
        "max_depth": trial.suggest_int("max_depth", 5, 40),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        "random_state": 42,
        "n_jobs": -1
    }

    model = RandomForestRegressor(**params)

    scores = cross_val_score(
        model,
        X_train_encoded,
        y_train,
        cv=kf,
        scoring="r2",
        n_jobs=-1
    )

    return scores.mean()

study_rf = optuna.create_study(
    direction="maximize",
    study_name="random_forest_rul_optimization"
)

study_rf.optimize(objective_rf, n_trials=50, show_progress_bar=True)

print("Best Random Forest Parameters:")
print(study_rf.best_params)
print(f"Best Random Forest CV R2: {study_rf.best_value:.4f}")

In [ ]:
best_rf_params = study_rf.best_params.copy()

best_rf = RandomForestRegressor(
    **best_rf_params,
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X_train_encoded, y_train)

y_pred_rf_tuned = best_rf.predict(X_test_encoded)

mae_rf_tuned = mean_absolute_error(y_test, y_pred_rf_tuned)
rmse_rf_tuned = np.sqrt(mean_squared_error(y_test, y_pred_rf_tuned))
r2_rf_tuned = r2_score(y_test, y_pred_rf_tuned)

print("\nTuned Random Forest Performance on Test Set")
print(f"MAE:  {mae_rf_tuned:.4f}")
print(f"RMSE: {rmse_rf_tuned:.4f}")
print(f"R2:   {r2_rf_tuned:.4f}")

### Observation
Hyperparameter tuning was attempted using Optuna.  
If the tuned model does not outperform the default Random Forest, the baseline configuration can still be selected as the final model because it is simpler and already performs strongly.

## CatBoost Optuna tuning

In [ ]:
import optuna
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Cross-validation strategy
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def objective_catboost(trial):
    params = {
        "loss_function": "RMSE",
        "eval_metric": "R2",
        "random_state": 42,
        "verbose": 0,

        # Core boosting params
        "iterations": trial.suggest_int("iterations", 300, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "depth": trial.suggest_int("depth", 4, 10),

        # Regularization / robustness
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 15.0),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 5.0),

        # Feature / sample control
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.6, 1.0),

        # Tree growth
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 50),
    }

    model = CatBoostRegressor(**params)

    scores = cross_val_score(
        model,
        X_train_encoded,
        y_train,
        cv=kf,
        scoring="r2",
        n_jobs=-1
    )

    return scores.mean()

study_cat = optuna.create_study(
    direction="maximize",
    study_name="catboost_rul_optimization"
)

study_cat.optimize(objective_catboost, n_trials=50, show_progress_bar=True)

print("Best CatBoost Parameters:")
print(study_cat.best_params)
print(f"Best CatBoost CV R2: {study_cat.best_value:.4f}")

In [ ]:
import pickle

pickle.dump(best_rf, open("best_rf.pkl", "wb"))

In [ ]:
pickle.dump(X_train_encoded.columns.tolist(), open("model_columns.pkl", "wb"))

In [ ]:
category_values = {
    "machine_type": sorted(data["machine_type"].dropna().unique().tolist()),
    "operating_mode": sorted(data["operating_mode"].dropna().unique().tolist())
}

pickle.dump(category_values, open("category_values.pkl", "wb"))

In [ ]:
num_cols = ['vibration_rms', 'temperature_motor', 'current_phase_avg',
            'pressure_level', 'rpm', 'hours_since_maintenance', 'ambient_temp']

cap_limits = {}

for col in num_cols:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    cap_limits[col] = {"lower": lower, "upper": upper}

pickle.dump(cap_limits, open("cap_limits.pkl", "wb"))

In [ ]:
import pickle

numeric_cols = [
    'vibration_rms',
    'temperature_motor',
    'current_phase_avg',
    'pressure_level',
    'rpm',
    'hours_since_maintenance',
    'ambient_temp'
]

input_ranges = {}

for col in numeric_cols:
    input_ranges[col] = {
        "min": float(data[col].min()),
        "max": float(data[col].max()),
        "q1": float(data[col].quantile(0.25)),
        "q3": float(data[col].quantile(0.75)),
        "median": float(data[col].median())
    }

pickle.dump(input_ranges, open("input_ranges.pkl", "wb"))

In [ ]:
best_cat_params = study_cat.best_params.copy()

best_cat = CatBoostRegressor(
    **best_cat_params,
    loss_function="RMSE",
    eval_metric="R2",
    random_state=42,
    verbose=0
)

best_cat.fit(X_train_encoded, y_train)

y_pred_cat_tuned = best_cat.predict(X_test_encoded)

mae_cat_tuned = mean_absolute_error(y_test, y_pred_cat_tuned)
rmse_cat_tuned = np.sqrt(mean_squared_error(y_test, y_pred_cat_tuned))
r2_cat_tuned = r2_score(y_test, y_pred_cat_tuned)

print("\nTuned CatBoost Performance on Test Set")
print(f"MAE:  {mae_cat_tuned:.4f}")
print(f"RMSE: {rmse_cat_tuned:.4f}")
print(f"R2:   {r2_cat_tuned:.4f}")

## LightGBM Optuna tuning

In [ ]:
from lightgbm import LGBMRegressor

def objective_lgbm(trial):
    params = {
        "objective": "regression",
        "metric": "rmse",
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1,

        # Core boosting params
        "n_estimators": trial.suggest_int("n_estimators", 300, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "num_leaves": trial.suggest_int("num_leaves", 20, 200),

        # Data control
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 80),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),

        # Regularization
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 20.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 20.0, log=True),

        # Split / leaf constraints
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 10.0, log=True),
    }

    model = LGBMRegressor(**params)

    scores = cross_val_score(
        model,
        X_train_encoded,
        y_train,
        cv=kf,
        scoring="r2",
        n_jobs=-1
    )

    return scores.mean()

study_lgbm = optuna.create_study(
    direction="maximize",
    study_name="lgbm_rul_optimization"
)

study_lgbm.optimize(objective_lgbm, n_trials=50, show_progress_bar=True)

print("Best LightGBM Parameters:")
print(study_lgbm.best_params)
print(f"Best LightGBM CV R2: {study_lgbm.best_value:.4f}")

In [ ]:
best_lgbm_params = study_lgbm.best_params.copy()

best_lgbm = LGBMRegressor(
    **best_lgbm_params,
    objective="regression",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

best_lgbm.fit(X_train_encoded, y_train)

y_pred_lgbm_tuned = best_lgbm.predict(X_test_encoded)

mae_lgbm_tuned = mean_absolute_error(y_test, y_pred_lgbm_tuned)
rmse_lgbm_tuned = np.sqrt(mean_squared_error(y_test, y_pred_lgbm_tuned))
r2_lgbm_tuned = r2_score(y_test, y_pred_lgbm_tuned)

print("\nTuned LightGBM Performance on Test Set")
print(f"MAE:  {mae_lgbm_tuned:.4f}")
print(f"RMSE: {rmse_lgbm_tuned:.4f}")
print(f"R2:   {r2_lgbm_tuned:.4f}")